In [13]:
# =========================================================
# Imports
# =========================================================
import os
import json
import time
import random
from datetime import datetime, timezone
from typing import List, Dict, Any, Optional, Tuple

import glob
import requests
import pandas as pd

In [2]:
# =========================================================
# CONFIG
# =========================================================

# 수집 대상 서브레딧 리스트 (원하는 만큼 추가 가능)
SUBREDDITS = [
    "Aion2",
    "Aion2Hub",
    "MMORPG",
    "lostark",
    "lostarkgame"
]

# Posts: top only, t=year only
POST_SORT = "top"
TOP_TIME_FILTER = "year"
TARGET_POSTS = 500

# Listing paging (100 * 5 = 500)
POSTS_LIMIT = 100
POSTS_PAGES = 5

# Comments
COLLECT_COMMENTS = True
COMMENT_SORT = "top"
COMMENTS_LIMIT = 500

# Rate limit 완화
SLEEP_BETWEEN_REQ = (0.1, 0.2)
SLEEP_BETWEEN_COMMENT_REQ = (0.1, 0.2)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36 "
                  "ResearchBot/1.0"
}

# Output base
BASE_OUTDIR = "reddit_out"
os.makedirs(BASE_OUTDIR, exist_ok=True)

In [4]:
# =========================================================
# Utils: sleep/state/http
# =========================================================

def jitter_sleep(a: float, b: float) -> None:
    time.sleep(random.uniform(a, b))

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def load_state(path: str) -> Dict[str, Any]:
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def save_state(path: str, obj: Dict[str, Any]) -> None:
    ensure_dir(os.path.dirname(path))
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False)

def safe_get_json(session: requests.Session, url: str, max_retries: int = 8) -> Optional[Any]:
    """
    - 무인증 reddit json은 429가 잦을 수 있음
    - Retry-After 있으면 따르고, 없으면 backoff
    - HTML로 떨어지는 경우(content-type)도 방어
    """
    backoff = 8
    for _ in range(max_retries):
        try:
            r = session.get(url, headers=HEADERS, timeout=30)
        except requests.RequestException:
            jitter_sleep(backoff, backoff + 1.5)
            backoff = min(backoff * 2, 180)
            continue

        if r.status_code == 200:
            ctype = (r.headers.get("content-type") or "").lower()
            if "application/json" not in ctype:
                jitter_sleep(backoff, backoff + 1.0)
                backoff = min(backoff * 2, 180)
                continue
            try:
                return r.json()
            except ValueError:
                return None

        if r.status_code == 429:
            ra = r.headers.get("Retry-After")
            wait = int(float(ra)) if ra else backoff
            time.sleep(wait + random.uniform(0, 1.0))
            print(f"429 received. Waited for {wait} seconds. Backoff now {backoff}.")
            backoff = min(backoff * 2, 180)
            continue

        # other errors
        jitter_sleep(backoff, backoff + 1.5)
        backoff = min(backoff * 2, 180)

    return None

# lostark 전용 필터 추가 (모바일과 PC게임 구분)
def is_lostark_mobile_post(text: str) -> bool:
    text = (text or "").lower()

    include_keywords = ["mobile", "모바일"]
    exclude_keywords = ["pc", "keyboard", "mouse", "steam"]

    # mobile 포함
    if not any(k in text for k in include_keywords):
        return False

    # pc 관련 키워드 제거
    if any(k in text for k in exclude_keywords):
        return False

    return True

In [5]:
# =========================================================
# Parsers
# =========================================================

def parse_post(child: Dict[str, Any], subreddit: str) -> Dict[str, Any]:
    d = child.get("data", {})
    created = int(d.get("created_utc", 0))
    permalink = d.get("permalink", "")

    return {
        "subreddit": subreddit,
        "sort_source": POST_SORT,
        "time_filter": TOP_TIME_FILTER,

        "id": d.get("id", ""),
        "fullname": d.get("name", ""),  # t3_xxx

        "created_utc": created,
        "created_dt_utc": datetime.fromtimestamp(created, tz=timezone.utc).isoformat() if created else "",

        "title": d.get("title", ""),
        "score": int(d.get("score", 0)),
        "upvote_ratio": d.get("upvote_ratio", None),
        "num_comments": int(d.get("num_comments", 0)),

        "permalink": ("https://www.reddit.com" + permalink) if permalink else "",
        "selftext": d.get("selftext", ""),
        "url": d.get("url", ""),
        "is_self": bool(d.get("is_self", False)),
    }


def flatten_top_level_comments(children: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    out = []
    for ch in children:
        if ch.get("kind") != "t1":
            continue
        d = ch.get("data", {})
        body = d.get("body")
        if not body or body in ("[deleted]", "[removed]"):
            continue
        out.append({
            "comment_id": d.get("id"),
            "comment_fullname": d.get("name", ""),
            "comment_author": d.get("author"),
            "comment_score": int(d.get("score", 0)),
            "comment_created_utc": int(d.get("created_utc", 0)),
            "comment_body": body,
        })
    return out

In [6]:
# =========================================================
# Step 1) Collect posts
# =========================================================

def fetch_posts_top_year(
    session: requests.Session,
    subreddit: str,
    pages: int = 5,
    limit: int = 100
) -> List[Dict[str, Any]]:
    base = (
        f"https://www.reddit.com/r/{subreddit}/{POST_SORT}.json"
        f"?t={TOP_TIME_FILTER}&limit={limit}&raw_json=1"
    )

    after = None
    rows: List[Dict[str, Any]] = []
    seen = set()

    for _ in range(pages):
        url = base + (f"&after={after}" if after else "")
        data = safe_get_json(session, url)
        if not data:
            break

        children = data.get("data", {}).get("children", [])
        if not children:
            break

        for ch in children:
            p = parse_post(ch, subreddit=subreddit)
            pid = p.get("id")

            # mobile 여부 컬럼 추가
            text = (p.get("title", "") + " " + p.get("selftext", ""))
            p["is_mobile_related"] = is_lostark_mobile_post(text)

            if subreddit in ["lostark", "lostarkgame"]:
                text = (p.get("title", "") + " " + p.get("selftext", ""))
                if not is_lostark_mobile_post(text):
                    continue

            if not pid or pid in seen:
                continue
            
            seen.add(pid)
            rows.append(p)

        after = data.get("data", {}).get("after")
        if not after:
            break

        jitter_sleep(*SLEEP_BETWEEN_REQ)

    return rows[:TARGET_POSTS]


def collect_posts_with_state(subreddit: str) -> pd.DataFrame:
    outdir = os.path.join(BASE_OUTDIR, f"reddit_out_{subreddit.lower()}")
    state_dir = os.path.join(outdir, "_state")
    ensure_dir(outdir)
    ensure_dir(state_dir)

    state_path = os.path.join(state_dir, "state_posts.json")
    st = load_state(state_path)

    if st.get("done") is True and isinstance(st.get("rows"), list) and len(st["rows"]) > 0:
        rows = st["rows"]
        print(f"[posts] loaded state r/{subreddit}: {len(rows)}")
    else:
        with requests.Session() as session:
            rows = fetch_posts_top_year(session, subreddit, pages=POSTS_PAGES, limit=POSTS_LIMIT)
        save_state(state_path, {"done": True, "rows": rows})
        print(f"[posts] fetched r/{subreddit}: {len(rows)}")

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df = df.drop_duplicates("id").copy()
    df["created_dt"] = pd.to_datetime(df["created_utc"], unit="s", utc=True, errors="coerce")
    df = df.dropna(subset=["created_dt"]).copy()

    df = df.sort_values(["score", "num_comments", "created_dt"], ascending=[False, False, False])

    df = df.head(TARGET_POSTS).reset_index(drop=True)
    return df

In [7]:
# =========================================================
# Step 2) Save posts
# =========================================================

def save_posts(subreddit: str, df_posts: pd.DataFrame) -> Tuple[str, str]:
    outdir = os.path.join(BASE_OUTDIR, f"reddit_out_{subreddit.lower()}")
    ensure_dir(outdir)

    posts_csv = os.path.join(outdir, f"posts_{subreddit}_{POST_SORT}_{TOP_TIME_FILTER}_{TARGET_POSTS}.csv")
    meta_json = os.path.join(outdir, f"meta_{subreddit}_{POST_SORT}_{TOP_TIME_FILTER}.json")

    if not df_posts.empty:
        df_posts.to_csv(posts_csv, index=False, encoding="utf-8-sig")

    meta = {
        "subreddit": subreddit,
        "post_sort": POST_SORT,
        "time_filter": TOP_TIME_FILTER,
        "target_posts": TARGET_POSTS,
        "collected_posts": int(len(df_posts)),
        "collected_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    with open(meta_json, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    print("[save] posts:", posts_csv, f"({len(df_posts)})")
    print("[save] meta :", meta_json)
    return posts_csv, meta_json

In [8]:
# =========================================================
# Step 3) Collect comments
# =========================================================

def fetch_comments_for_post(
    session: requests.Session,
    permalink: str,
    limit: int = 100
) -> List[Dict[str, Any]]:
    if not permalink:
        return []

    url = permalink.rstrip("/") + f".json?limit={limit}&sort={COMMENT_SORT}&raw_json=1"
    data = safe_get_json(session, url)
    if not data or not isinstance(data, list) or len(data) < 2:
        return []

    children = data[1].get("data", {}).get("children", [])
    return flatten_top_level_comments(children)


def collect_comments_with_state(subreddit: str, df_posts: pd.DataFrame) -> pd.DataFrame:
    outdir = os.path.join(BASE_OUTDIR, f"reddit_out_{subreddit.lower()}")
    state_dir = os.path.join(outdir, "_state")
    ensure_dir(outdir)
    ensure_dir(state_dir)

    state_path = os.path.join(state_dir, "state_comments.json")
    st = load_state(state_path)

    done_ids = set(st.get("done_post_ids", []))
    rows = st.get("rows", [])

    total = len(df_posts)
    if total == 0:
        return pd.DataFrame()

    with requests.Session() as session:
        for idx, r in enumerate(df_posts.itertuples(index=False), 1):
            post_id = getattr(r, "id", "")
            if not post_id or post_id in done_ids:
                continue

            permalink = getattr(r, "permalink", "")
            comments = fetch_comments_for_post(session, permalink, limit=COMMENTS_LIMIT)

            for c in comments:
                rows.append({
                    "subreddit": subreddit,
                    "post_sort": POST_SORT,
                    "post_time_filter": TOP_TIME_FILTER,

                    "post_id": post_id,
                    "post_created_utc": int(getattr(r, "created_utc", 0)),
                    "post_title": getattr(r, "title", ""),
                    "post_score": int(getattr(r, "score", 0)),
                    "post_num_comments": int(getattr(r, "num_comments", 0)),
                    "post_permalink": permalink,

                    "comment_sort": COMMENT_SORT,
                    "comment_id": c.get("comment_id"),
                    "comment_fullname": c.get("comment_fullname"),
                    "comment_author": c.get("comment_author"),
                    "comment_score": int(c.get("comment_score", 0)),
                    "comment_created_utc": int(c.get("comment_created_utc", 0)),
                    "comment_body": c.get("comment_body", ""),
                })

            done_ids.add(post_id)
            save_state(state_path, {"done_post_ids": list(done_ids), "rows": rows})

            print(f"[comments] r/{subreddit} {idx}/{total} post_id={post_id} -> {len(comments)} top-level comments")
            jitter_sleep(*SLEEP_BETWEEN_COMMENT_REQ)

    dfc = pd.DataFrame(rows)
    if dfc.empty:
        return dfc

    dfc = dfc.drop_duplicates(["post_id", "comment_id"]).reset_index(drop=True)
    return dfc

In [9]:
# =========================================================
# Step 4) Save comments
# =========================================================

def save_comments(subreddit: str, df_comments: pd.DataFrame) -> str:
    outdir = os.path.join(BASE_OUTDIR, f"reddit_out_{subreddit.lower()}")
    ensure_dir(outdir)

    comments_csv = os.path.join(
        outdir,
        f"comments_{subreddit}_from_{TARGET_POSTS}posts_{COMMENT_SORT}_limit{COMMENTS_LIMIT}.csv"
    )
    if not df_comments.empty:
        df_comments.to_csv(comments_csv, index=False, encoding="utf-8-sig")

    print("[save] comments:", comments_csv, f"({len(df_comments)})")
    return comments_csv

In [ ]:
# =========================================================
# RUN
# =========================================================

for sub in SUBREDDITS:
    print("\n" + "="*90)
    print(f"SUBREDDIT: r/{sub} | posts={TARGET_POSTS} | sort=top | t=year | comments<= {COMMENTS_LIMIT}/post")
    print("="*90)

    # Posts
    df_posts = collect_posts_with_state(sub)
    posts_csv, meta_json = save_posts(sub, df_posts)

    print("\n[posts summary]")
    print(" - collected:", len(df_posts))
    if len(df_posts) > 0:
        print(" - newest  :", df_posts["created_dt"].max())
        print(" - oldest  :", df_posts["created_dt"].min())

    # Comments
    if COLLECT_COMMENTS:
        print("\nCollecting comments...")
        df_comments = collect_comments_with_state(sub, df_posts)
        comments_csv = save_comments(sub, df_comments)

        print("\n[comments summary]")
        print(" - total rows:", len(df_comments))
        if len(df_comments) > 0:
            n_posts = df_comments["post_id"].nunique()
            print(" - posts with >=1 comment row:", n_posts)
            print(" - avg comment rows/post    :", round(len(df_comments) / max(n_posts, 1), 2))

print("\n==== DONE ====")
print("BASE_OUTDIR:", BASE_OUTDIR)

In [14]:
# Reddit posts CSV 폴더 경로
BASE_DIR = "reddit_out"  # 레딧 저장 폴더
OUT_FILE = os.path.join(BASE_DIR, "reddit_all_posts_combined.csv")

# 모든 reddit_out_* 폴더에서 posts CSV 찾기
all_csv_files = []
for folder in glob.glob(os.path.join(BASE_DIR, "reddit_out_*")):
    csv_files = glob.glob(os.path.join(folder, "posts_*.csv"))
    all_csv_files.extend(csv_files)

print(f"Found {len(all_csv_files)} posts CSV files:")
for f in all_csv_files:
    print(" -", f)

# CSV 파일 하나로 합치기
dfs = []
for f in all_csv_files:
    df = pd.read_csv(f, encoding="utf-8-sig")
    # 서브레딧 이름을 파일명에서 추출
    subreddit_name = os.path.basename(f).split("_")[1]
    df["subreddit"] = subreddit_name
    # Reddit은 해외로 고정
    df["region"] = "해외"
    dfs.append(df)

if dfs:
    df_all = pd.concat(dfs, axis=0, ignore_index=True)
else:
    df_all = pd.DataFrame()

# 중복 제거, 정리
if not df_all.empty:
    df_all = df_all.drop_duplicates(subset=["id"]).reset_index(drop=True)
    print(f"Combined dataframe shape: {df_all.shape}")
    # 최근 생성일 기준으로 정렬
    if "created_utc" in df_all.columns:
        df_all = df_all.sort_values("created_utc", ascending=False)

Found 4 posts CSV files:
 - reddit_out\reddit_out_aion2\posts_Aion2_top_year_500.csv
 - reddit_out\reddit_out_aion2hub\posts_Aion2Hub_top_year_500.csv
 - reddit_out\reddit_out_lostarkgame\posts_lostarkgame_top_year_500.csv
 - reddit_out\reddit_out_mmorpg\posts_MMORPG_top_year_500.csv
Combined dataframe shape: (1181, 18)


In [16]:
df_all

,subreddit,sort_source,time_filter,id,fullname,created_utc,created_dt_utc,title,score,upvote_ratio,num_comments,permalink,selftext,url,is_self,is_mobile_related,created_dt,region
280,Aion2,top,year,1s0urce,t3_1s0urce,1774207585,2026-03-22T19:26:25+00:00,AION2 DPS Meter major update available!,18,1.00,12,https://www.reddit.com/r/Aion2/comments/1s0urc...,I've been hard at work on a major update for t...,https://www.reddit.com/r/Aion2/comments/1s0urc...,True,False,2026-03-22 19:26:25+00:00,해외
864,MMORPG,top,year,1s0hpzf,t3_1s0hpzf,1774173490,2026-03-22T09:58:10+00:00,"I miss this game so, so much",477,0.93,128,https://www.reddit.com/r/MMORPG/comments/1s0hp...,NaN,https://i.redd.it/s9v6chdlkkqg1.jpeg,False,False,2026-03-22 09:58:10+00:00,해외
639,Aion2Hub,top,year,1s0c6qd,t3_1s0c6qd,1774153237,2026-03-22T04:20:37+00:00,Kaisinel's Illusion — Boss Guide,1,1.00,0,https://www.reddit.com/r/Aion2Hub/comments/1s0...,Hopefully I covered everything you need to kno...,https://aion2.app/guides/nightmare-kaisinel,False,False,2026-03-22 04:20:37+00:00,해외
265,Aion2,top,year,1s02byk,t3_1s02byk,1774126067,2026-03-21T20:47:47+00:00,Bro this game is so popular!,19,0.69,50,https://www.reddit.com/r/Aion2/comments/1s02by...,Look at all these new players,https://v.redd.it/lnkpwiekngqg1,False,False,2026-03-21 20:47:47+00:00,해외
769,MMORPG,top,year,1rzz2te,t3_1rzz2te,1774118190,2026-03-21T18:36:30+00:00,Hello darkness my old friend,784,0.95,91,https://www.reddit.com/r/MMORPG/comments/1rzz2...,NaN,https://www.reddit.com/gallery/1rzz2te,False,False,2026-03-21 18:36:30+00:00,해외
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1001,MMORPG,top,year,1jv7892,t3_1jv7892,1744209151,2025-04-09T14:32:31+00:00,A MMO Release Timeline,304,0.96,119,https://www.reddit.com/r/MMORPG/comments/1jv78...,**TLDR:** I built [The MMO Roadmap](https://ro...,https://roadmap.nephi-labs.com/,False,False,2025-04-09 14:32:31+00:00,해외
901,MMORPG,top,year,1jstzfn,t3_1jstzfn,1743945696,2025-04-06T13:21:36+00:00,Every upcoming MMORPG,419,0.94,421,https://www.reddit.com/r/MMORPG/comments/1jstz...,Hey guys. I wanted to compile a list of all th...,https://www.reddit.com/r/MMORPG/comments/1jstz...,True,False,2025-04-06 13:21:36+00:00,해외
1074,MMORPG,top,year,1jpb2j9,t3_1jpb2j9,1743552562,2025-04-02T00:09:22+00:00,HEALING FROG is still looking for his friend...,256,0.77,73,https://www.reddit.com/r/MMORPG/comments/1jpb2...,NaN,https://www.reddit.com/gallery/1jpb2j9,False,False,2025-04-02 00:09:22+00:00,해외
767,MMORPG,top,year,1jp4tn2,t3_1jp4tn2,1743536656,2025-04-01T19:44:16+00:00,WoW is still one of the best MMOs,802,0.78,549,https://www.reddit.com/r/MMORPG/comments/1jp4t...,"After all these years, WoW offers a good pve e...",https://www.reddit.com/r/MMORPG/comments/1jp4t...,True,False,2025-04-01 19:44:16+00:00,해외


In [17]:
# title이 비어 있는 행 조회
empty_title_df = df_all[df_all["title"].isna() | (df_all["title"].str.strip() == "")]

print(f"title이 비어있는 행 수: {len(empty_title_df)}")
if not empty_title_df.empty:
    # 서브레딧, id, 생성 시간, URL 같이 확인
    print(empty_title_df[["subreddit", "id", "created_utc", "url"]])

title이 비어있는 행 수: 0


In [18]:
# 최종 CSV 저장
df_all.to_csv(OUT_FILE, index=False, encoding="utf-8-sig")
print(f"Saved combined posts CSV to {OUT_FILE}")

Saved combined posts CSV to reddit_out\reddit_all_posts_combined.csv


In [19]:
# title, selftext만 추출해서 CSV로 저장
translate_csv_path = "reddit_title_selftext_for_translation.csv"
df_translate = df_all[["title", "selftext"]].copy()
df_translate.to_csv(translate_csv_path, index=False, encoding="utf-8-sig")
print(f"[save] title, selftext 추출 CSV: {translate_csv_path}")

[save] title, selftext 추출 CSV: reddit_title_selftext_for_translation.csv


In [20]:
df_translate

,title,selftext
280,AION2 DPS Meter major update available!,I've been hard at work on a major update for t...
864,"I miss this game so, so much",NaN
639,Kaisinel's Illusion — Boss Guide,Hopefully I covered everything you need to kno...
265,Bro this game is so popular!,Look at all these new players
769,Hello darkness my old friend,NaN
...,...,...
1001,A MMO Release Timeline,**TLDR:** I built [The MMO Roadmap](https://ro...
901,Every upcoming MMORPG,Hey guys. I wanted to compile a list of all th...
1074,HEALING FROG is still looking for his friend...,NaN
767,WoW is still one of the best MMOs,"After all these years, WoW offers a good pve e..."


In [23]:
# 번역 결과 불러오기
# 번역 후 CSV에는 동일하게 title, selftext 컬럼으로 저장했다고 가정
translated_csv_path = "reddit_title_selftext_for_translation.csv"
df_translated = pd.read_csv(translated_csv_path, encoding="utf-8-sig")
df_translated

,제목,본문
0,아이온2 DPS 미터기 대규모 업데이트 완료!,무료 DPS 미터기 업데이트를 위해 한동안 매달려 왔는데요. 무엇보다 데미지 측정 ...
1,이 게임이 너무나도 그립네요.,NaN
2,카이시넬의 환영 — 보스 공략,"이 보스에 대해 알아야 할 모든 정보를 다뤘기를 바랍니다.한마디 덧붙이자면, 템플러..."
3,"와, 이 게임 진짜 인기 많네!",새로 온 유저들 좀 봐
4,"안녕 어둠아, 나의 오랜 친구여",NaN
...,...,...
1177,Every upcoming MMORPG,Hey guys. I wanted to compile a list of all th...
1178,HEALING FROG is still looking for his friend...,NaN
1179,WoW is still one of the best MMOs,"After all these years, WoW offers a good pve e..."
1180,As a new player who is snooping around the sub...,NaN


In [26]:
# 컬럼 이름 변경
df_translated = df_translated.rename(columns={
    "제목": "title",
    "본문": "selftext"
})

In [32]:
before_drop = len(df_translated)
df_translated = df_translated[df_translated["title"].notna() & (df_translated["title"].str.strip() != "")]
after_drop = len(df_translated)
print(f"[info] title 비어있는 행 삭제: {before_drop - after_drop}행 제거, 남은 행 {after_drop}")

[info] title 비어있는 행 삭제: 1행 제거, 남은 행 1181


In [33]:
df_translated

,title,selftext
0,아이온2 DPS 미터기 대규모 업데이트 완료!,무료 DPS 미터기 업데이트를 위해 한동안 매달려 왔는데요. 무엇보다 데미지 측정 ...
1,이 게임이 너무나도 그립네요.,NaN
2,카이시넬의 환영 — 보스 공략,"이 보스에 대해 알아야 할 모든 정보를 다뤘기를 바랍니다.한마디 덧붙이자면, 템플러..."
3,"와, 이 게임 진짜 인기 많네!",새로 온 유저들 좀 봐
4,"안녕 어둠아, 나의 오랜 친구여",NaN
...,...,...
1176,A MMO Release Timeline,**TLDR:** I built [The MMO Roadmap](https://ro...
1177,Every upcoming MMORPG,Hey guys. I wanted to compile a list of all th...
1178,HEALING FROG is still looking for his friend...,NaN
1179,WoW is still one of the best MMOs,"After all these years, WoW offers a good pve e..."


In [38]:
# df_all에서 기존 title/selftext 삭제
for col in ["title", "selftext"]:
    if col in df_all.columns:
        df_all.drop(columns=col, inplace=True)
print(f"[info] 기존 title/selftext 삭제, df_all columns: {df_all.columns.tolist()}")

# df_translated의 title/selftext를 df_all과 같은 인덱스로 합치기
df_translated = df_translated[["title", "selftext"]].reset_index(drop=True)
df_all = df_all.reset_index(drop=True)
df_all = pd.concat([df_all, df_translated], axis=1)
print(f"[info] df_translated 병합 완료, df_all shape: {df_all.shape}")

[info] 기존 title/selftext 삭제, df_all columns: ['subreddit', 'sort_source', 'time_filter', 'id', 'fullname', 'created_utc', 'created_dt_utc', 'score', 'upvote_ratio', 'num_comments', 'permalink', 'url', 'is_self', 'is_mobile_related', 'created_dt', 'region']
[info] df_translated 병합 완료, df_all shape: (1181, 18)


In [39]:
df_all

,subreddit,sort_source,time_filter,id,fullname,created_utc,created_dt_utc,score,upvote_ratio,num_comments,permalink,url,is_self,is_mobile_related,created_dt,region,title,selftext
0,Aion2,top,year,1s0urce,t3_1s0urce,1774207585,2026-03-22T19:26:25+00:00,18,1.00,12,https://www.reddit.com/r/Aion2/comments/1s0urc...,https://www.reddit.com/r/Aion2/comments/1s0urc...,True,False,2026-03-22 19:26:25+00:00,해외,아이온2 DPS 미터기 대규모 업데이트 완료!,무료 DPS 미터기 업데이트를 위해 한동안 매달려 왔는데요. 무엇보다 데미지 측정 ...
1,MMORPG,top,year,1s0hpzf,t3_1s0hpzf,1774173490,2026-03-22T09:58:10+00:00,477,0.93,128,https://www.reddit.com/r/MMORPG/comments/1s0hp...,https://i.redd.it/s9v6chdlkkqg1.jpeg,False,False,2026-03-22 09:58:10+00:00,해외,이 게임이 너무나도 그립네요.,NaN
2,Aion2Hub,top,year,1s0c6qd,t3_1s0c6qd,1774153237,2026-03-22T04:20:37+00:00,1,1.00,0,https://www.reddit.com/r/Aion2Hub/comments/1s0...,https://aion2.app/guides/nightmare-kaisinel,False,False,2026-03-22 04:20:37+00:00,해외,카이시넬의 환영 — 보스 공략,"이 보스에 대해 알아야 할 모든 정보를 다뤘기를 바랍니다.한마디 덧붙이자면, 템플러..."
3,Aion2,top,year,1s02byk,t3_1s02byk,1774126067,2026-03-21T20:47:47+00:00,19,0.69,50,https://www.reddit.com/r/Aion2/comments/1s02by...,https://v.redd.it/lnkpwiekngqg1,False,False,2026-03-21 20:47:47+00:00,해외,"와, 이 게임 진짜 인기 많네!",새로 온 유저들 좀 봐
4,MMORPG,top,year,1rzz2te,t3_1rzz2te,1774118190,2026-03-21T18:36:30+00:00,784,0.95,91,https://www.reddit.com/r/MMORPG/comments/1rzz2...,https://www.reddit.com/gallery/1rzz2te,False,False,2026-03-21 18:36:30+00:00,해외,"안녕 어둠아, 나의 오랜 친구여",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1176,MMORPG,top,year,1jv7892,t3_1jv7892,1744209151,2025-04-09T14:32:31+00:00,304,0.96,119,https://www.reddit.com/r/MMORPG/comments/1jv78...,https://roadmap.nephi-labs.com/,False,False,2025-04-09 14:32:31+00:00,해외,A MMO Release Timeline,**TLDR:** I built [The MMO Roadmap](https://ro...
1177,MMORPG,top,year,1jstzfn,t3_1jstzfn,1743945696,2025-04-06T13:21:36+00:00,419,0.94,421,https://www.reddit.com/r/MMORPG/comments/1jstz...,https://www.reddit.com/r/MMORPG/comments/1jstz...,True,False,2025-04-06 13:21:36+00:00,해외,Every upcoming MMORPG,Hey guys. I wanted to compile a list of all th...
1178,MMORPG,top,year,1jpb2j9,t3_1jpb2j9,1743552562,2025-04-02T00:09:22+00:00,256,0.77,73,https://www.reddit.com/r/MMORPG/comments/1jpb2...,https://www.reddit.com/gallery/1jpb2j9,False,False,2025-04-02 00:09:22+00:00,해외,HEALING FROG is still looking for his friend...,NaN
1179,MMORPG,top,year,1jp4tn2,t3_1jp4tn2,1743536656,2025-04-01T19:44:16+00:00,802,0.78,549,https://www.reddit.com/r/MMORPG/comments/1jp4t...,https://www.reddit.com/r/MMORPG/comments/1jp4t...,True,False,2025-04-01 19:44:16+00:00,해외,WoW is still one of the best MMOs,"After all these years, WoW offers a good pve e..."


In [40]:
# 덮어쓴 df_all 다시 저장
df_all_path = "reddit_all_posts_translated.csv"
df_all.to_csv(df_all_path, index=False, encoding="utf-8-sig")
print(f"[save] 번역 후 df_all CSV: {df_all_path}")

[save] 번역 후 df_all CSV: reddit_all_posts_translated.csv
